In [ ]:
import os
import pandas as pd
import numpy as np
import random
import math
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

semilla = 42
random.seed(semilla)
np.random.seed(semilla)
tf.random.set_seed(semilla)

path = os.path.join(os.getcwd(), "ibm.us.csv")
dataset = pd.read_csv(path)
dataset.head()

#70% train, 15% validation, 15% test
n = len(dataset)

train = dataset[:int(n*0.7)][["promedioDia"]].to_numpy()
validation = dataset[int(n*0.7):int(n*0.85)][["promedioDia"]].to_numpy()
test = dataset[int(n*0.85):][["promedioDia"]].to_numpy()

print("Conjunto de entrenamiento: ",train[:5])
print("Conjunto de validación: ",validation[:5])
print("Conjunto de prueba: ",test[:5])

#Hallar la media y desviación del entrenamiento
media = train.mean()
desvest  = train.std()

#Normalizar los 3 subconjuntos respecto a los valores calculados
train = (train - media) / desvest
validation   = (validation   - media) / desvest
test  = (test  - media) / desvest

train2D = train.reshape(-1, 1)
validation2D = validation.reshape(-1, 1)
test2D = test.reshape(-1, 1)

In [ ]:
def graficaeMAE(history, titulo, ax=None):
  if ax is None:
    ax = plt.gca()

  loss = history.history["mae"]
  val_loss = history.history["val_mae"]
  epochs = range(1, len(loss) + 1)

  ax.plot(epochs, loss, "o-", label="MAE Entrenamiento")
  ax.plot(epochs, val_loss, "-", label="MAE Validación")
  ax.set_title(titulo)
  ax.set_xlabel("Época")
  ax.set_ylabel("MAE")
  ax.legend()

def graficarPrediccion(prediccion, y, titulo, ax=None):
  rango=(20, 70)
  
  if ax is None:
    ax = plt.gca()
        
  start, end = rango
  ax.plot(y[start:end], label="Valor Real")
  ax.plot(prediccion[start:end], label="Predicción")
  ax.set_title(titulo)
  ax.set_xlabel("Índice")
  ax.set_ylabel("Valor")
  ax.legend()

In [ ]:
#Ventanas de 3 días con muestras diarias
sampling_rate = 1
sequence_length = 3
delay = 1 
batch_size = 32
cantNeuronas = 15
epocas = 42

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

#Definir los nuevo dataset
datasetTrainApilado = keras.utils.timeseries_dataset_from_array(
  data = train2D[:-delay],
  targets=train2D[delay:],
  sampling_rate=sampling_rate,
  sequence_length=3,
  shuffle=True,
  batch_size=32,
  start_index=0
)

datasetValidationApilado = keras.utils.timeseries_dataset_from_array(
  data = validation2D[:-delay],
  targets=validation2D[delay:],
  sampling_rate=sampling_rate,
  sequence_length=3,
  shuffle=False,
  batch_size=32,
  start_index=0
)

datasetTestApilado = keras.utils.timeseries_dataset_from_array(
  data = test2D[:-delay],
  targets=test2D[delay:],
  sampling_rate=sampling_rate,
  sequence_length=3,
  shuffle=False,
  batch_size=32,
  start_index=0
)

#Crear el modelo 
inputs = keras.Input(shape=(3, 1))
x = layers.LSTM(15)(inputs)
outputs = layers.Dense(1)(x)
modelo = keras.Model(inputs, outputs)

#Entrenar el modelo
cb = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)]
modelo.compile(optimizer="rmsprop", loss="mse", metrics=["mae"])    

history = modelo.fit(
  datasetTrainApilado,
  epochs=42,
  validation_data=datasetValidationApilado,
  callbacks=cb,
  verbose=0   
)

#Evaluar el modelo
mae = modelo.evaluate(datasetTestApilado, verbose=0)[1]
y = np.concatenate([y for x, y in datasetTestApilado], axis=0)
pred = modelo.predict(datasetTestApilado)  
graficaeMAE(history, f"MAE de Entrenamiento y Validación - LSTM - Mejores Resultados", axes[0])
graficarPrediccion(pred, y, f"Predicción vs Valor Real - LSTM - Mejores Resultados", axes[1])

print(f"MAE de prueba - Mejores Resultados): {mae:.4f}")

plt.tight_layout()
plt.show()